# 03 Model Training and Prediction

This notebook trains machine learning models to predict dissolved oxygen (DO) in mg/L and percentage for multiple future horizons.

Models used:
- Linear Regression
- Random Forest
- XGBoost

Prediction horizons:
15, 30, 45, 60, 75, 90, 105, and 120 minutes.

Cell 1 — Markdown
Cell 2 — Imports
Cell 3 — Paths and Settings
Cell 4 — Load Final Modelling Dataset
Cell 5 — Check Features and Targets
Cell 6 — Helper Functions
Cell 7 — Train One Target Function
Cell 8 — Train All Models
Cell 9 — Save Metrics and Predictions
Cell 10 — Best Models by Horizon
Cell 11 — Create Final Forecast Table
Cell 12 — Model Performance Summary
Cell 13 — Forecast Preview
Cell 14 - Summary

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [2]:
DATA_PATH = "../data/processed/aquasensor_final.csv"
MODELS_DIR = "../models"
OUTPUT_DIR = "../data/processed"

PERFORMANCE_PATH = "../data/processed/do_prediction_model_performance.csv"
ALL_PREDICTIONS_PATH = "../data/processed/all_model_predictions.csv"
FORECASTS_PATH = "../data/processed/river_do_forecasts.csv"

HORIZONS = [
    "15min", "30min", "45min", "60min",
    "75min", "90min", "105min", "120min"
]

NUMERIC_FEATURES = [
    "temperature",
    "air_temperature_c",
    "sunshine_wm2",
    "hour",
    "dissolved_oxygen_mgl",
    "dissolved_oxygen_pct",
    "pollution_alert",
    "anomaly_type",
    "season_proxy",
]

CATEGORICAL_FEATURES = [
    "sensor_id",
    "sensor_name",
]

In [3]:
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"], low_memory=False)

df["sensor_id"] = df["sensor_id"].astype(str)
df["sensor_name"] = df["sensor_name"].astype(str)

print("Loaded modelling data")
print("Rows:", len(df))
print("Date range:", df["timestamp"].min(), "to", df["timestamp"].max())

df.head()

Loaded modelling data
Rows: 107974
Date range: 2023-10-30 00:00:00 to 2026-06-04 21:13:40


,timestamp,sensor_id,sensor_name,temperature,dissolved_oxygen_mgl,dissolved_oxygen_pct,air_temperature_c,sunshine_wm2,cloud_cover_pct,hour,...,do_mgl_next_60min,do_pct_next_60min,do_mgl_next_75min,do_pct_next_75min,do_mgl_next_90min,do_pct_next_90min,do_mgl_next_105min,do_pct_next_105min,do_mgl_next_120min,do_pct_next_120min
0,2025-06-01 00:04:48,941115,Derwent 13-50,13.0,9.7,91.2,11.3,0.0,17.0,0.0,...,9.7,90.5,9.6,90.4,9.6,90.2,9.6,90.1,9.6,89.9
1,2025-06-01 00:16:44,941115,Derwent 13-50,12.9,9.7,91.1,11.3,0.0,17.0,0.0,...,9.6,90.4,9.6,90.2,9.6,90.1,9.6,89.9,9.6,89.8
2,2025-06-01 00:28:41,941115,Derwent 13-50,12.9,9.7,90.9,11.3,0.0,17.0,0.0,...,9.6,90.2,9.6,90.1,9.6,89.9,9.6,89.8,9.6,89.7
3,2025-06-01 00:40:37,941115,Derwent 13-50,12.8,9.7,90.7,11.3,0.0,17.0,0.0,...,9.6,90.1,9.6,89.9,9.6,89.8,9.6,89.7,9.6,89.6
4,2025-06-01 00:52:33,941115,Derwent 13-50,12.8,9.7,90.5,11.3,0.0,17.0,0.0,...,9.6,89.9,9.6,89.8,9.6,89.7,9.6,89.6,9.6,89.5


In [4]:
print("Numeric features:")
print(NUMERIC_FEATURES)

print("\nCategorical features:")
print(CATEGORICAL_FEATURES)

print("\nAvailable target columns:")
target_cols = [c for c in df.columns if c.startswith("do_mgl_next_") or c.startswith("do_pct_next_")]
print(target_cols)

Numeric features:
['temperature', 'air_temperature_c', 'sunshine_wm2', 'hour', 'dissolved_oxygen_mgl', 'dissolved_oxygen_pct', 'pollution_alert', 'anomaly_type', 'season_proxy']

Categorical features:
['sensor_id', 'sensor_name']

Available target columns:
['do_mgl_next_15min', 'do_pct_next_15min', 'do_mgl_next_30min', 'do_pct_next_30min', 'do_mgl_next_45min', 'do_pct_next_45min', 'do_mgl_next_60min', 'do_pct_next_60min', 'do_mgl_next_75min', 'do_pct_next_75min', 'do_mgl_next_90min', 'do_pct_next_90min', 'do_mgl_next_105min', 'do_pct_next_105min', 'do_mgl_next_120min', 'do_pct_next_120min']


In [5]:
def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERIC_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        ]
    )


def get_models():
    return {
        "Linear Regression": LinearRegression(),

        "Random Forest": RandomForestRegressor(
            n_estimators=100,
            max_depth=12,
            random_state=42,
            n_jobs=-1,
        ),

        "XGBoost": XGBRegressor(
            n_estimators=250,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1,
        ),
    }


def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2


def safe_filename_text(text):
    return (
        text.lower()
        .replace(" ", "_")
        .replace("%", "pct")
        .replace("/", "_per_")
    )

In [6]:
def train_single_target(df, target_col, target_type, horizon):
    model_df = df.dropna(subset=[target_col]).copy()

    X = model_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
    y = model_df[target_col]

    meta = model_df[
        [
            "timestamp",
            "sensor_id",
            "sensor_name",
            "dissolved_oxygen_mgl",
            "dissolved_oxygen_pct",
            "pollution_alert",
            "anomaly_type",
        ]
    ].copy()

    X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
        X,
        y,
        meta,
        test_size=0.2,
        shuffle=False,
    )

    metrics_rows = []
    prediction_rows = []

    print("\n" + "=" * 75)
    print(f"TARGET: {target_type} | HORIZON: {horizon}")
    print("=" * 75)

    for model_name, model in get_models().items():
        print(f"Training {model_name}...")

        pipeline = Pipeline(
            steps=[
                ("preprocessor", build_preprocessor()),
                ("model", model),
            ]
        )

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        mae, rmse, r2 = calculate_metrics(y_test, y_pred)

        print(f"MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

        metrics_rows.append(
            {
                "target_type": target_type,
                "horizon": horizon,
                "target_column": target_col,
                "model": model_name,
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2,
                "train_rows": len(X_train),
                "test_rows": len(X_test),
            }
        )

        model_file = (
            f"{safe_filename_text(model_name)}_"
            f"{safe_filename_text(target_type)}_"
            f"{horizon}.pkl"
        )

        model_path = os.path.join(MODELS_DIR, model_file)
        joblib.dump(pipeline, model_path)

        pred_df = meta_test.copy()
        pred_df["target_type"] = target_type
        pred_df["horizon"] = horizon
        pred_df["model"] = model_name
        pred_df["actual_value"] = y_test.values
        pred_df["predicted_value"] = y_pred
        pred_df["absolute_error"] = np.abs(y_test.values - y_pred)

        prediction_rows.append(pred_df)

    metrics_df = pd.DataFrame(metrics_rows)
    predictions_df = pd.concat(prediction_rows, ignore_index=True)

    return metrics_df, predictions_df

In [7]:
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_metrics = []
all_predictions = []

for horizon in HORIZONS:
    targets = [
        ("DO mg/L", f"do_mgl_next_{horizon}"),
        ("DO %", f"do_pct_next_{horizon}"),
    ]

    for target_type, target_col in targets:
        metrics_df, predictions_df = train_single_target(
            df=df,
            target_col=target_col,
            target_type=target_type,
            horizon=horizon,
        )

        all_metrics.append(metrics_df)
        all_predictions.append(predictions_df)

final_metrics = pd.concat(all_metrics, ignore_index=True)
final_predictions = pd.concat(all_predictions, ignore_index=True)

print("Model training complete")


TARGET: DO mg/L | HORIZON: 15min
Training Linear Regression...
MAE=0.0594, RMSE=0.1428, R2=0.9941
Training Random Forest...
MAE=0.1886, RMSE=0.4252, R2=0.9479
Training XGBoost...
MAE=0.4995, RMSE=0.6943, R2=0.8610

TARGET: DO % | HORIZON: 15min
Training Linear Regression...
MAE=0.4861, RMSE=1.2644, R2=0.9913
Training Random Forest...
MAE=0.5781, RMSE=1.5382, R2=0.9871
Training XGBoost...
MAE=1.6659, RMSE=2.6940, R2=0.9603

TARGET: DO mg/L | HORIZON: 30min
Training Linear Regression...
MAE=0.0851, RMSE=0.2001, R2=0.9885
Training Random Forest...
MAE=0.2450, RMSE=0.5104, R2=0.9249
Training XGBoost...
MAE=0.4884, RMSE=0.6606, R2=0.8742

TARGET: DO % | HORIZON: 30min
Training Linear Regression...
MAE=0.7085, RMSE=1.8154, R2=0.9820
Training Random Forest...
MAE=0.8271, RMSE=1.9941, R2=0.9783
Training XGBoost...
MAE=1.3900, RMSE=3.0407, R2=0.9494

TARGET: DO mg/L | HORIZON: 45min
Training Linear Regression...
MAE=0.1085, RMSE=0.2475, R2=0.9823
Training Random Forest...
MAE=0.2431, RMSE=0.50

In [8]:
final_metrics.to_csv(PERFORMANCE_PATH, index=False)
final_predictions.to_csv(ALL_PREDICTIONS_PATH, index=False)

print("Saved model performance to:", PERFORMANCE_PATH)
print("Saved all predictions to:", ALL_PREDICTIONS_PATH)

Saved model performance to: ../data/processed/do_prediction_model_performance.csv
Saved all predictions to: ../data/processed/all_model_predictions.csv


In [9]:
best_models = (
    final_metrics.sort_values("RMSE")
    .groupby(["target_type", "horizon"])
    .head(1)[["target_type", "horizon", "model", "MAE", "RMSE", "R2"]]
)

best_models

,target_type,horizon,model,MAE,RMSE,R2
0,DO mg/L,15min,Linear Regression,0.059432,0.142753,0.994124
6,DO mg/L,30min,Linear Regression,0.085146,0.200089,0.988456
12,DO mg/L,45min,Linear Regression,0.108538,0.247535,0.982333
18,DO mg/L,60min,Linear Regression,0.129906,0.282753,0.976948
24,DO mg/L,75min,Linear Regression,0.152506,0.334699,0.967701
30,DO mg/L,90min,Linear Regression,0.174986,0.388585,0.956464
36,DO mg/L,105min,Linear Regression,0.195606,0.427643,0.947274
42,DO mg/L,120min,Linear Regression,0.214726,0.456126,0.940018
3,DO %,15min,Linear Regression,0.486125,1.264440,0.991258
9,DO %,30min,Linear Regression,0.708503,1.815417,0.981980


In [10]:
forecast_tables = []

for target_type in ["DO mg/L", "DO %"]:
    subset = final_predictions[final_predictions["target_type"] == target_type]

    best_model = (
        subset.groupby("model")["absolute_error"]
        .mean()
        .sort_values()
        .index[0]
    )

    best_subset = subset[subset["model"] == best_model].copy()

    pivot = best_subset.pivot_table(
        index=[
            "timestamp",
            "sensor_id",
            "sensor_name",
            "dissolved_oxygen_mgl",
            "dissolved_oxygen_pct",
            "pollution_alert",
            "anomaly_type",
        ],
        columns="horizon",
        values="predicted_value",
        aggfunc="first",
    ).reset_index()

    if target_type == "DO mg/L":
        pivot = pivot.rename(
            columns={
                "15min": "predicted_do_mgl_15min",
                "30min": "predicted_do_mgl_30min",
                "45min": "predicted_do_mgl_45min",
                "60min": "predicted_do_mgl_60min",
                "75min": "predicted_do_mgl_75min",
                "90min": "predicted_do_mgl_90min",
                "105min": "predicted_do_mgl_105min",
                "120min": "predicted_do_mgl_120min",
            }
        )
    else:
        pivot = pivot.rename(
            columns={
                "15min": "predicted_do_pct_15min",
                "30min": "predicted_do_pct_30min",
                "45min": "predicted_do_pct_45min",
                "60min": "predicted_do_pct_60min",
                "75min": "predicted_do_pct_75min",
                "90min": "predicted_do_pct_90min",
                "105min": "predicted_do_pct_105min",
                "120min": "predicted_do_pct_120min",
            }
        )

    forecast_tables.append(pivot)

river_do_forecasts = forecast_tables[0].merge(
    forecast_tables[1],
    on=[
        "timestamp",
        "sensor_id",
        "sensor_name",
        "dissolved_oxygen_mgl",
        "dissolved_oxygen_pct",
        "pollution_alert",
        "anomaly_type",
    ],
    how="outer",
)

river_do_forecasts = river_do_forecasts.rename(
    columns={
        "timestamp": "latest_sensor_timestamp",
        "dissolved_oxygen_mgl": "current_do_mgl",
        "dissolved_oxygen_pct": "current_do_pct",
    }
)

river_do_forecasts.to_csv(FORECASTS_PATH, index=False)

print("Saved final DO forecasts to:", FORECASTS_PATH)
print("Rows:", len(river_do_forecasts))

river_do_forecasts.head()

Saved final DO forecasts to: ../data/processed/river_do_forecasts.csv
Rows: 21597


horizon,latest_sensor_timestamp,sensor_id,sensor_name,current_do_mgl,current_do_pct,pollution_alert,anomaly_type,predicted_do_mgl_105min,predicted_do_mgl_120min,predicted_do_mgl_15min,...,predicted_do_mgl_75min,predicted_do_mgl_90min,predicted_do_pct_105min,predicted_do_pct_120min,predicted_do_pct_15min,predicted_do_pct_30min,predicted_do_pct_45min,predicted_do_pct_60min,predicted_do_pct_75min,predicted_do_pct_90min
0,2025-11-13 03:25:20,sensor022,Derwent 13,11.0,97.6,0,0,11.014636,11.016540,NaN,...,NaN,11.012938,97.971543,98.024195,NaN,NaN,NaN,NaN,NaN,97.917929
1,2025-11-13 03:37:18,sensor022,Derwent 13,11.0,97.7,0,0,11.011916,11.013514,NaN,...,11.009206,11.010577,98.048957,98.098977,NaN,NaN,NaN,97.895969,97.946383,97.998037
2,2025-11-13 03:49:17,sensor022,Derwent 13,11.0,97.2,0,0,11.025515,11.028645,11.005726,...,11.019109,11.022384,97.661886,97.725066,97.260332,97.328754,97.397917,97.466926,97.532045,97.597495
3,2025-11-13 04:01:21,sensor022,Derwent 13,11.0,97.3,0,0,11.022313,11.026677,11.003776,...,11.014595,11.018401,97.621429,97.686427,97.321158,97.357630,97.402459,97.453573,97.504505,97.561128
4,2025-11-13 04:13:19,sensor022,Derwent 13,11.0,97.1,0,0,11.024366,11.029162,11.004053,...,11.015904,11.020082,97.446459,97.516493,97.122793,97.161934,97.210286,97.265564,97.320544,97.381566


In [11]:
summary = final_metrics.sort_values(["target_type", "horizon", "RMSE"])
summary.head(30)

,target_type,horizon,target_column,model,MAE,RMSE,R2,train_rows,test_rows
39,DO %,105min,do_pct_next_105min,Linear Regression,1.782251,3.993427,0.912818,86362,21591
40,DO %,105min,do_pct_next_105min,Random Forest,1.813071,4.908877,0.868265,86362,21591
41,DO %,105min,do_pct_next_105min,XGBoost,1.918140,5.479629,0.835850,86362,21591
45,DO %,120min,do_pct_next_120min,Linear Regression,1.970592,4.235015,0.901954,86360,21590
46,DO %,120min,do_pct_next_120min,Random Forest,1.986854,5.315220,0.845559,86360,21590
47,DO %,120min,do_pct_next_120min,XGBoost,1.958826,5.579973,0.829791,86360,21590
3,DO %,15min,do_pct_next_15min,Linear Regression,0.486125,1.264440,0.991258,86376,21595
4,DO %,15min,do_pct_next_15min,Random Forest,0.578053,1.538240,0.987062,86376,21595
5,DO %,15min,do_pct_next_15min,XGBoost,1.665940,2.694045,0.960315,86376,21595
9,DO %,30min,do_pct_next_30min,Linear Regression,0.708503,1.815417,0.981980,86374,21594


In [12]:
river_do_forecasts.head(10)

horizon,latest_sensor_timestamp,sensor_id,sensor_name,current_do_mgl,current_do_pct,pollution_alert,anomaly_type,predicted_do_mgl_105min,predicted_do_mgl_120min,predicted_do_mgl_15min,...,predicted_do_mgl_75min,predicted_do_mgl_90min,predicted_do_pct_105min,predicted_do_pct_120min,predicted_do_pct_15min,predicted_do_pct_30min,predicted_do_pct_45min,predicted_do_pct_60min,predicted_do_pct_75min,predicted_do_pct_90min
0,2025-11-13 03:25:20,sensor022,Derwent 13,11.0,97.6,0,0,11.014636,11.016540,NaN,...,NaN,11.012938,97.971543,98.024195,NaN,NaN,NaN,NaN,NaN,97.917929
1,2025-11-13 03:37:18,sensor022,Derwent 13,11.0,97.7,0,0,11.011916,11.013514,NaN,...,11.009206,11.010577,98.048957,98.098977,NaN,NaN,NaN,97.895969,97.946383,97.998037
2,2025-11-13 03:49:17,sensor022,Derwent 13,11.0,97.2,0,0,11.025515,11.028645,11.005726,...,11.019109,11.022384,97.661886,97.725066,97.260332,97.328754,97.397917,97.466926,97.532045,97.597495
3,2025-11-13 04:01:21,sensor022,Derwent 13,11.0,97.3,0,0,11.022313,11.026677,11.003776,...,11.014595,11.018401,97.621429,97.686427,97.321158,97.357630,97.402459,97.453573,97.504505,97.561128
4,2025-11-13 04:13:19,sensor022,Derwent 13,11.0,97.1,0,0,11.024366,11.029162,11.004053,...,11.015904,11.020082,97.446459,97.516493,97.122793,97.161934,97.210286,97.265564,97.320544,97.381566
5,2025-11-13 04:25:19,sensor022,Derwent 13,11.0,97.2,0,0,11.021646,11.026136,11.004027,...,11.013923,11.017721,97.523873,97.591276,97.218421,97.253991,97.299111,97.351372,97.403412,97.461675
6,2025-11-13 04:37:18,sensor022,Derwent 13,11.0,97.4,0,0,11.016207,11.020084,11.003975,...,11.009961,11.012998,97.678702,97.740840,97.409678,97.438103,97.476761,97.522989,97.569147,97.621891
7,2025-11-13 04:49:17,sensor022,Derwent 13,11.0,97.4,0,0,11.016207,11.020084,11.003975,...,11.009961,11.012998,97.678702,97.740840,97.409678,97.438103,97.476761,97.522989,97.569147,97.621891
8,2025-11-13 05:01:16,sensor022,Derwent 13,11.0,97.0,0,0,11.021753,11.025810,11.004082,...,11.014547,11.018069,97.297697,97.355108,97.023106,97.058570,97.100080,97.146205,97.192669,97.243652
9,2025-11-13 05:13:15,sensor022,Derwent 13,11.0,97.3,0,0,11.013594,11.016732,11.004004,...,11.008605,11.010985,97.529939,97.579455,97.309991,97.334739,97.366555,97.403630,97.441272,97.483978


## Summary

This notebook trained Linear Regression, Random Forest, and XGBoost models for dissolved oxygen prediction.

The models predicted DO in both mg/L and percentage for 15, 30, 45, 60, 75, 90, 105, and 120 minutes ahead.

The outputs generated were:

- `do_prediction_model_performance.csv`
- `all_model_predictions.csv`
- `river_do_forecasts.csv`
- trained model files saved in the `models/` folder